# Регуляризованная регрессия: Ridge, Lasso и ElasticNet

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Регуляризованная регрессия».

## Что делает метод

Регуляризованная регрессия — это линейная модель со штрафом на величину коэффициентов. Штраф стабилизирует оценки при многочисленных и коррелированных предикторах. Ridge (L2-штраф) сжимает коэффициенты к нулю, не обнуляя их; Lasso (L1-штраф) зануляет часть коэффициентов и тем самым отбирает признаки; ElasticNet комбинирует оба штрафа.

Метод применим, когда предикторов много, они коррелируют между собой или их число сопоставимо с числом наблюдений — в таких условиях обычный метод наименьших квадратов даёт неустойчивые коэффициенты.

В этом блокноте мы сравним Ridge, Lasso и ElasticNet, подберём силу штрафа кросс-валидацией и разберём отбор признаков. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте, что вам нужно написать отчёт, но есть жёсткий лимит объёма. Вы не можете рассказать обо всём и с большими подробностями — нужно выбрать самое важное и написать сжато. Регуляризация работает точно так же: она добавляет в модель «штраф за сложность», вынуждая её работать с небольшими, ограниченными коэффициентами и оставлять только значимые предикторы.

Зачем это нужно? Когда предикторов много (сотни генов, тысячи спектральных каналов, десятки социально-экономических индикаторов), обычная линейная регрессия начинает «подстраиваться под шум»: она запоминает случайные особенности обучающих данных и плохо работает на новых. Это называется **переобучением**. Регуляризация намеренно «сдерживает» модель, снижая переобучение.

Три варианта штрафа:
- **Ridge (L2)** — штрафует за сумму квадратов коэффициентов. Сжимает все коэффициенты к нулю, но не обнуляет их. Хорош при мультиколлинеарности (коррелирующих предикторах).
- **Lasso (L1)** — штрафует за сумму модулей коэффициентов. Часть коэффициентов обнуляет полностью — это автоматический отбор признаков.
- **ElasticNet** — комбинирует оба штрафа: устойчивее Lasso при группах коррелирующих предикторов.

**Ключевые термины:**
- **Переобучение** — модель слишком хорошо подогнана под обучающие данные и плохо обобщается на новые.
- **Регуляризация** — штраф на сложность модели, снижающий переобучение.
- **Гиперпараметр** — параметр, который задаёт не сама модель, а исследователь до обучения (например, сила штрафа α). Подбирается кросс-валидацией.
- **Кросс-валидация** — метод оценки качества модели: данные разбиваются на несколько частей, модель поочерёдно обучается на одних частях и проверяется на оставшейся.
- **Мультиколлинеарность** — сильная корреляция между предикторами, из-за которой коэффициенты обычной регрессии становятся неустойчивыми.

## 1. Установка библиотек

В среде Google Colab перечисленные библиотеки, как правило, уже установлены. Ячейка ниже гарантирует их наличие, в том числе при локальном запуске.

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib

## 2. Единый стиль графиков

Все графики в блокнотах проекта оформляются в едином визуальном стиле сайта «ИИ для учёных»: общая палитра, шрифты, толщины линий и сетка. Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py` — отдельный файл загружать не нужно. Вызов `apply_viz_style()` настраивает matplotlib один раз на весь блокнот.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",  # фон полотна
    "node_fill":  "#eef1f6",  # заливка блоков
    "node_text":  "#0e1726",  # основной текст
    "edge":       "#4d5e78",  # линии, оси, подписи
    "grid":       "#dce2ec",  # сетка координат
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],  # цвета рядов
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Сгенерируем синтетический набор с 60 предикторами, из которых лишь 8 действительно влияют на отклик, а часть предикторов коррелирует между собой. Это типичная ситуация, в которой регуляризация необходима: истинная структура разрежена, а предикторов больше, чем сигнала.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n_samples, n_features, n_informative = 200, 60, 8

X = rng.normal(size=(n_samples, n_features))
# вводим корреляцию: часть признаков - зашумлённые копии информативных
for j in range(n_informative, 2 * n_informative):
    X[:, j] = X[:, j - n_informative] + 0.3 * rng.normal(size=n_samples)

true_coef = np.zeros(n_features)
true_coef[:n_informative] = rng.uniform(2, 5, n_informative)
y = X @ true_coef + rng.normal(scale=2.0, size=n_samples)

feature_names = [f'x{j}' for j in range(n_features)]
X = pd.DataFrame(X, columns=feature_names)
print(f'Наблюдений: {n_samples}, предикторов: {n_features}, '
      f'из них информативных: {n_informative}')
X.iloc[:5, :8]

### Наглядный «ага»-эксперимент: эффект силы штрафа на коэффициенты Lasso

Прежде чем переходить к подбору параметров, посмотрим, что происходит с коэффициентами при разных значениях силы штрафа α. Это ключевая интуиция: при слабом штрафе модель ведёт себя как обычная регрессия, при сильном — обнуляет всё больше коэффициентов, оставляя только самые важные предикторы.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

# Подготовим данные (используем уже созданный X из раздела 3)
sc = StandardScaler()
X_sc = sc.fit_transform(X.to_numpy())
y_arr = y

alphas = np.logspace(-3, 1, 200)
coef_paths = []
for a in alphas:
    lasso = Lasso(alpha=a, max_iter=20000)
    lasso.fit(X_sc, y_arr)
    coef_paths.append(lasso.coef_.copy())
coef_paths = np.array(coef_paths)

fig, ax = plt.subplots(figsize=(11.0, 5.8))
colors_path = get_palette(n_informative)
for j in range(n_informative):
    ax.semilogx(alphas, coef_paths[:, j],
                color=VIZ['series'][j % 4],
                linewidth=1.8, alpha=0.9,
                label=f'x{j} (информативный)')
for j in range(n_informative, min(n_informative + 5, n_features)):
    ax.semilogx(alphas, coef_paths[:, j],
                color=VIZ['edge'], linewidth=0.8,
                alpha=0.5, linestyle='--')

# Аннотация для одного «лишнего» предиктора
ax.annotate('Неинформативные предикторы\n(быстро обнуляются)',
            xy=(alphas[80], coef_paths[80, n_informative]),
            xytext=(alphas[30], -0.8),
            arrowprops=dict(arrowstyle='->', color=VIZ['edge']),
            color=VIZ['edge'], fontsize=10)

ax.axvline(alphas[100], color=VIZ['series'][2], linestyle=':',
           alpha=0.6, label='умеренный штраф')
ax.set_xlabel('Сила штрафа α (логарифмическая шкала)')
ax.set_ylabel('Значение коэффициента')
ax.set_title('Lasso: как штраф обнуляет коэффициенты (путь регуляризации)')
ax.legend(loc='upper right', fontsize=9, ncol=2)
fig.tight_layout()
plt.show()

**Как читать этот график (путь регуляризации).** По горизонтали — сила штрафа α в логарифмическом масштабе (слева — слабый штраф, справа — сильный). По вертикали — значения коэффициентов. При слабом штрафе (левый край) коэффициенты информативных предикторов ненулевые. По мере усиления штрафа неинформативные предикторы (серые пунктирные линии) обнуляются первыми; информативные «держатся» дольше, но и они в итоге обнуляются при очень сильном штрафе. Именно эта точка — правильный баланс — и подбирается кросс-валидацией в следующем разделе.

## 4. Применение метода

Разделим данные и подберём силу штрафа кросс-валидацией. Классы `RidgeCV`, `LassoCV` и `ElasticNetCV` автоматически перебирают значения параметра регуляризации α и выбирают лучшее по качеству предсказания на отложенных фолдах кросс-валидации. Признаки предварительно стандартизуются — это обязательно, иначе штраф будет несправедливо сильнее наказывать признаки с большим масштабом.

**Почему мы сравниваем три метода?** Ridge, Lasso и ElasticNet по-разному устроены, и лучший вариант зависит от структуры данных. Посмотрим на их поведение бок о бок.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

models = {
    'Ridge (L2)': make_pipeline(StandardScaler(),
        RidgeCV(alphas=np.logspace(-2, 3, 40))),
    'Lasso (L1)': make_pipeline(StandardScaler(),
        LassoCV(cv=5, random_state=42, max_iter=10000)),
    'ElasticNet': make_pipeline(StandardScaler(),
        ElasticNetCV(cv=5, random_state=42, max_iter=10000)),
}
for name, m in models.items():
    m.fit(X_train, y_train)
print('Все три модели обучены.')

### Сравнение качества и отбор признаков

Следующая ячейка сравнивает три модели по двум критериям:
- **R² на тестовой выборке** — качество прогноза на данных, которых модель не видела.
- **Число ненулевых коэффициентов** — какое число предикторов фактически используется. Lasso и ElasticNet должны оставить лишь информативные из 60 исходных.

In [ ]:
from sklearn.metrics import r2_score

rows = []
for name, m in models.items():
    est = m.steps[-1][1]
    coef = est.coef_
    rows.append({
        'Модель': name,
        'R2 на тесте': round(r2_score(y_test, m.predict(X_test)), 3),
        'Ненулевых коэффициентов': int(np.sum(np.abs(coef) > 1e-6)),
    })
pd.DataFrame(rows)

### Визуальное сравнение коэффициентов

На графике ниже сопоставим оценённые коэффициенты Ridge и Lasso с истинными значениями. Идеальная модель должна иметь крупные коэффициенты у первых 8 информативных предикторов (слева от пунктирной линии) и нулевые — у остальных 52.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(13.5, 5.6))
idx = np.arange(n_features)

ax.bar(idx - 0.3, true_coef, width=0.3,
       color=VIZ['series'][3], label='истинные коэффициенты')
ax.bar(idx, models['Lasso (L1)'].steps[-1][1].coef_, width=0.3,
       color=VIZ['series'][0], label='Lasso')
ax.bar(idx + 0.3, models['Ridge (L2)'].steps[-1][1].coef_, width=0.3,
       color=VIZ['series'][1], label='Ridge')
ax.axvline(n_informative - 0.5, color=VIZ['series'][2],
           linestyle='--', label='граница информативных')
ax.set_xlabel('Номер предиктора')
ax.set_ylabel('Значение коэффициента')
ax.set_title('Оценённые коэффициенты против истинных')
ax.legend()
fig.tight_layout()
plt.show()

**Как читать этот график.** Каждая группа из трёх столбиков — один предиктор. Серые столбики — истинные значения коэффициентов (известны нам, поскольку данные синтетические). Синие — коэффициенты Lasso, зелёные — Ridge. Пунктирная вертикальная линия отделяет 8 информативных предикторов (слева) от 52 неинформативных (справа). Lasso должен обнулить большинство столбиков справа, тогда как Ridge лишь сжимает их, но не обнуляет. Чем ближе синие и зелёные столбики к серым, тем точнее модель восстановила истинные коэффициенты.

## 5. Интерпретация результата

- **R² на тесте**: показывает, насколько точно модель предсказывает новые наблюдения. При коррелированных предикторах регуляризованные модели обычно превосходят обычный МНК.
- **Число ненулевых коэффициентов**: Ridge сохраняет все предикторы, Lasso и ElasticNet зануляют неинформативные — это встроенный отбор признаков.
- **График коэффициентов**: Lasso концентрирует сигнал на немногих предикторах, Ridge распределяет его более равномерно, что устойчивее при сильной корреляции.
- **Выбор метода**: Ridge — когда важны все предикторы и есть мультиколлинеарность; Lasso — когда нужен компактный набор признаков; ElasticNet — компромисс, устойчивый при группах коррелированных признаков.

Сила штрафа подобрана кросс-валидацией; при смене данных её следует подбирать заново.

## 6. Попробуйте на своих данных

Замените синтетический набор своими данными — особенно если предикторов много или они коррелируют.

1. Загрузите файл в Colab через вкладку «Файлы».
2. Снимите комментарии в ячейке ниже и укажите имя файла и столбец с откликом.
3. Выполните ячейки разделов 4 и 5.

## 7. Поэкспериментируйте сами

1. **Измените число информативных признаков.** В разделе 3 смените `n_informative = 8` на `n_informative = 20`. Как изменится доля признаков, которые Lasso обнуляет? Когда задача становится труднее для отбора признаков?

2. **Добавьте обычную линейную регрессию.** Обучите `LinearRegression()` на тех же данных и добавьте её R² в сравнительную таблицу. При 60 предикторах и только 200 наблюдениях обычная регрессия должна показать заметно худший результат — это и есть переобучение.

3. **Увеличьте корреляцию предикторов.** В разделе 3 смените коэффициент шума `0.3` на `0.05` (очень сильная корреляция). Посмотрите, как Lasso ведёт себя при коррелированных признаках в сравнении с ElasticNet.

In [ ]:
# --- Шаблон загрузки своих данных ---
# df = pd.read_csv('my_data.csv')
# target_column = 'отклик'
#
# y = df[target_column].to_numpy()
# X = df.drop(columns=[target_column]).select_dtypes('number')
# feature_names = list(X.columns)
# n_features = X.shape[1]
#
# print(f'Наблюдений: {X.shape[0]}, предикторов: {n_features}')
# Далее повторите ячейки раздела 4.

## 8. Краткие выводы

- Регуляризация добавляет к функции потерь штраф на величину коэффициентов, что «сдерживает» модель и снижает переобучение при большом числе предикторов.
- Ridge сжимает все коэффициенты к нулю, не обнуляя их — полезен при мультиколлинеарности. Lasso обнуляет часть коэффициентов — встроенный отбор признаков. ElasticNet — компромисс при коррелированных группах признаков.
- «Путь регуляризации» (график коэффициентов от силы штрафа) наглядно показывает, какие предикторы важны: они обнуляются последними.
- Сила штрафа α — гиперпараметр; его нужно подбирать кросс-валидацией. Нельзя использовать тестовую выборку ни при подборе α, ни при стандартизации признаков.
- Обычные p-значения и доверительные интервалы для регуляризованных коэффициентов некорректны; если нужен статистический вывод — используйте специальные методы (debiased Lasso) или бутстрэп.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. Вы увеличили силу штрафа α в модели Lasso в 100 раз. На графике «путь регуляризации» видно, что все коэффициенты обратились в ноль, а R² на тесте упал до нуля. Что произошло с балансом смещение/разброс и какой признак поведения модели это характеризует?
2. Lasso обнулил 52 из 60 предикторов, однако два из восьми оставшихся — это коррелирующие копии одного и того же информативного признака. Почему Lasso выбирает произвольно один из группы коллинеарных признаков и что предпочтительнее применить в такой ситуации?
3. Коллега использует тестовую выборку для выбора оптимального α (перебирает α, каждый раз оценивая R² на тесте, и берёт лучший). Почему эта процедура неправомерна и как она должна быть организована корректно?

<details>
<summary>Показать ориентиры для ответов</summary>

1. При слишком большом α штраф подавляет все коэффициенты — модель вырождается в константу (предсказывает среднее обучающей выборки). Это чрезмерное смещение: разброс предсказаний близок к нулю, но систематическая ошибка огромна. Оптимальный α находится в точке компромисса между смещением и разбросом, которую определяет кросс-валидация.
2. Lasso при коллинеарности стабильно обнуляет все признаки группы, кроме одного, — но выбор конкретного нестабилен: зависит от случайных флуктуаций в данных. ElasticNet (L1 + L2) распределяет вес между всеми коррелирующими признаками группы, что обеспечивает более стабильный и интерпретируемый отбор.
3. Использование тестовой выборки для выбора гиперпараметра нарушает её независимость: оценка R² становится оптимистически смещённой и не отражает обобщающей способности. Правильная процедура: α подбирается кросс-валидацией исключительно на обучающей выборке (например, через `LassoCV`), а тестовая выборка используется один раз — для финальной оценки выбранной модели.
</details>